<a href="https://colab.research.google.com/github/utkuayten/CS401-soffritto/blob/main/optuna_GAT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/utkuayten/CS401-soffritto

fatal: destination path 'CS401-soffritto' already exists and is not an empty directory.


In [2]:
%pwd

'/content'

In [3]:
%ls

CS401-soffritto/  sample_data/


In [4]:
%cd CS401-soffritto/

/content/CS401-soffritto


In [5]:
%ls

CS401-soffritto/                 optuna_informerV2_BOCO.ipynb
dataset_feature_selection.ipynb  PatchTST/
GAT/                             README.md
GenomicBert/                     run_model.ipynb
iTransformer/                    soffritto/
optuna_BOCO_H1.csv               train_w_parameters.ipynb
optuna_GAT.ipynb                 transofritto/
optuna_informer_BOC.ipynb        transofritto_v2/
optuna_informer_BOCO.ipynb       wavelet_informer.ipynb
optuna_informerV2_BOC.ipynb


In [6]:
!pip install optuna

In [7]:
!pip install -U pyg-lib torch-scatter torch-sparse torch-cluster torch-spline-conv \
  -f https://data.pyg.org/whl/torch-2.9.0+cu126.html

!pip install -U torch-geometric

Looking in links: https://data.pyg.org/whl/torch-2.9.0+cu126.html
ERROR: Could not find a version that satisfies the requirement pyg-lib (from versions: none)
ERROR: No matching distribution found for pyg-lib


In [8]:
!python3 GAT/optuna_tune_gat_intracell.py

[I 2026-01-08 14:43:55,775] A new study created in memory with name: gat_intracell_tune
epoch 001 | train_KL=0.628653 | test_KL=0.573796 | best=0.573796
epoch 010 | train_KL=0.451112 | test_KL=0.393391 | best=0.393391
epoch 020 | train_KL=0.238659 | test_KL=0.206947 | best=0.206947
epoch 030 | train_KL=0.135824 | test_KL=0.121248 | best=0.121248
epoch 040 | train_KL=0.093361 | test_KL=0.089239 | best=0.089239
epoch 050 | train_KL=0.074951 | test_KL=0.076441 | best=0.076441
epoch 060 | train_KL=0.066519 | test_KL=0.069195 | best=0.069195
epoch 070 | train_KL=0.061156 | test_KL=0.064285 | best=0.064285
epoch 080 | train_KL=0.057425 | test_KL=0.061071 | best=0.061071
epoch 090 | train_KL=0.054227 | test_KL=0.058059 | best=0.058059
epoch 100 | train_KL=0.051580 | test_KL=0.055656 | best=0.055656
epoch 110 | train_KL=0.049298 | test_KL=0.053558 | best=0.053558
epoch 120 | train_KL=0.047121 | test_KL=0.051698 | best=0.051698
epoch 130 | train_KL=0.045256 | test_KL=0.049906 | best=0.049906
ep

In [9]:
import numpy as np
import torch
import torch.nn.functional as F

def load_first_npz_array(npz_path: str) -> np.ndarray:
    z = np.load(npz_path)
    return z[z.files[8]]  # exactly like your code

def load_gat_probs_from_npz(npz_path: str) -> np.ndarray:
    z = np.load(npz_path)
    return z["probs"]  # saved as probs in your GAT/predictions/*.npz

def kl_pq_mean_and_per_sample(p_true: torch.Tensor, q_pred: torch.Tensor):
    """
    KL(P||Q) where P=true labels (prob), Q=predictions (prob).
    Shapes: (N, K)
    Returns: (kl_per_sample: (N,), kl_mean: scalar)
    """
    kl_elem = F.kl_div(q_pred.log(), p_true, reduction="none")  # (N, K)
    kl_per_sample = kl_elem.sum(dim=1)                          # (N,)
    kl_mean = kl_per_sample.mean()                              # scalar
    return kl_per_sample, kl_mean


cell_lines = ['H1']
for cell in cell_lines:
    pred = torch.from_numpy(load_gat_probs_from_npz(f"GAT/predictions/{cell}_predictions.npz")).to(torch.float64)
    labels = torch.from_numpy(load_first_npz_array(f"GAT/data/{cell}_labels.npz")).to(torch.float64)

    assert pred.shape == labels.shape, f"Shape mismatch: pred{pred.shape} vs labels{labels.shape}"

    kl_each, kl_avg = kl_pq_mean_and_per_sample(labels, pred)
    print(f"For cell {cell} KL loss: {kl_avg:.5f}")
    print("First 5 KL values:", kl_each[:5], "\n")

FileNotFoundError: [Errno 2] No such file or directory: 'GAT/predictions/H1_predictions.npz'

In [ ]:
import torch

print("torch:", torch.__version__)
print("torch cuda build:", torch.version.cuda)  # None => CPU-only torch
print("cuda available:", torch.cuda.is_available())
print("cuda device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu0:", torch.cuda.get_device_name(0))

In [ ]:
!python -c "import torch; print('torch', torch.__version__); print('torch.version.cuda', torch.version.cuda); print('cuda available', torch.cuda.is_available())"

In [11]:
!nvidia-smi -L

GPU 0: NVIDIA L4 (UUID: GPU-658ca9f9-aac3-ed24-9073-b7f16faf2061)
